[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C17_Classical_NLP_Course/05_nlp_pipeline/05_nlp_pipeline.ipynb)

# 05 · NLP 流水线与评测（纯 numpy 从零）

目标：**不调 sklearn**，从零实现一条完整文本分类管线——**分词归一化** → **TF-IDF** → **朴素贝叶斯 / 逻辑回归** → **精确率/召回率/F1**，并验证每步正确（TF-IDF 压低普遍词、NB 对拍手算、F1 对拍公式）。

路线：分词归一化 → 词袋 → **TF-IDF** → **朴素贝叶斯(对数空间+平滑)** → **F1 评测** → ✏️ 练习 → 📖 答案 → 🧪 真实文本分类胶囊。

> 心智模型：**文档→向量→类别→F1**。TF-IDF 自动压低 the/of、抬高有区分度的词; F1 在不平衡数据下比 accuracy 诚实。

## 1 · 分词与归一化

把原始文本切成 token 并统一形态：小写、去标点、（可选）去停用词。**铁律**：训练和测试用同一个预处理函数。

In [ ]:
import numpy as np
import re
from collections import Counter, defaultdict

STOP = {'the', 'a', 'an', 'is', 'are', 'of', 'to', 'and', 'in', 'on'}

def tokenize(text, remove_stop=True):
    '''小写 + 按非字母切 + (可选)去停用词。'''
    toks = re.findall(r"[a-z]+", text.lower())
    if remove_stop:
        toks = [t for t in toks if t not in STOP]
    return toks

doc = 'The Cat sat on the MAT! A cat ran.'
print('原文:', doc)
print('分词(去停用):', tokenize(doc))
print('分词(留停用):', tokenize(doc, remove_stop=False))
assert tokenize(doc) == ['cat', 'sat', 'mat', 'cat', 'ran']
assert 'the' not in tokenize(doc) and 'the' in tokenize(doc, remove_stop=False)
print('✅ 分词归一化：小写+去标点+去停用词, 大小写已合并')

## 2 · 词袋向量化

把 token 序列变成「词→计数」向量（忽略词序）。先在训练语料上**建词表**，再把每篇文档映射成计数向量。

In [ ]:
def build_vocab(docs):
    vocab = sorted({t for d in docs for t in tokenize(d)})
    return {w: i for i, w in enumerate(vocab)}

def bow_vector(text, w2i):
    v = np.zeros(len(w2i))
    for t in tokenize(text):
        if t in w2i:                  # OOV 词忽略
            v[w2i[t]] += 1
    return v

docs = ['the cat sat on the mat', 'the dog ran in the park', 'a cat and a dog']
w2i = build_vocab(docs)
print('词表:', list(w2i.keys()))
bow = bow_vector('the cat sat with a cat', w2i)
print('BoW(the cat sat with a cat):', bow.astype(int), '(with/the/a 被去停或OOV)')
assert bow[w2i['cat']] == 2, 'cat 出现 2 次'
assert bow.sum() == 3, 'cat,cat,sat 三个有效词'
print('✅ 词袋向量化：文档 -> 计数向量(忽略词序)')

## 3 · TF-IDF：压低普遍词、抬高区分词

$\mathrm{tfidf}(t,d)=\mathrm{tf}(t,d)\times\log\frac{N}{\mathrm{df}(t)}$。在所有文档都出现的词 IDF≈0(被压低); 稀有词 IDF 大(被抬高)。先 fit(算 IDF), 再 transform。

In [ ]:
def fit_idf(docs, w2i):
    '''算每个词的 IDF = log(N / df)。'''
    N = len(docs)
    df = np.zeros(len(w2i))
    for d in docs:
        seen = set(tokenize(d))
        for t in seen:
            if t in w2i:
                df[w2i[t]] += 1
    return np.log(N / np.maximum(df, 1))            # 避免除零

def tfidf_vector(text, w2i, idf):
    tf = bow_vector(text, w2i)
    return tf * idf

# 用一个 'common' 出现在所有文档、'rare' 只在一篇的语料看 TF-IDF 效果
docs2 = ['common rare apple', 'common banana', 'common cherry']
w2i2 = build_vocab(docs2)
idf2 = fit_idf(docs2, w2i2)
print('IDF(common) =', round(idf2[w2i2['common']], 3), '(在所有文档 -> ≈0)')
print('IDF(rare)   =', round(idf2[w2i2['rare']], 3), '(只在1篇 -> 大)')
assert idf2[w2i2['common']] < idf2[w2i2['rare']], '普遍词 IDF 应低于稀有词'
assert np.isclose(idf2[w2i2['common']], 0.0), 'common 在全部文档 -> IDF=log(3/3)=0'
vec = tfidf_vector('common rare', w2i2, idf2)
assert vec[w2i2['common']] == 0.0, 'common 的 TF-IDF 被压到 0'
assert vec[w2i2['rare']] > 0
print('✅ TF-IDF: 普遍词 common 被压到 0, 稀有词 rare 被抬高')

## 4 · 朴素贝叶斯分类器（对数空间 + 平滑）

$\hat c=\arg\max_c[\log P(c)+\sum_t \log P(t|c)]$。$P(t|c)$ 用类 $c$ 里词频 + 拉普拉斯平滑。对数空间防下溢。从零训练并对拍手算。

In [ ]:
def train_naive_bayes(docs, labels, alpha=1.0):
    '''返回 (log_prior, log_likelihood, w2i, classes)。'''
    w2i = build_vocab(docs); V = len(w2i)
    classes = sorted(set(labels))
    log_prior = {}; log_lik = {}
    for c in classes:
        c_docs = [d for d, l in zip(docs, labels) if l == c]
        log_prior[c] = np.log(len(c_docs) / len(docs))
        # 类 c 的词计数
        wc = np.zeros(V)
        for d in c_docs:
            wc += bow_vector(d, w2i)
        # 拉普拉斯平滑: (count + alpha) / (total + alpha*V)
        log_lik[c] = np.log((wc + alpha) / (wc.sum() + alpha * V))
    return log_prior, log_lik, w2i, classes

def predict_nb(text, log_prior, log_lik, w2i, classes):
    bow = bow_vector(text, w2i)
    scores = {c: log_prior[c] + float(bow @ log_lik[c]) for c in classes}
    return max(scores, key=scores.get), scores

# 玩具: 体育 vs 科技
train_docs = ['game team score win', 'team player game goal',
              'computer data code software', 'data model code compute']
train_labels = ['sport', 'sport', 'tech', 'tech']
lp, ll, w2i_nb, cls = train_naive_bayes(train_docs, train_labels)
pred, sc = predict_nb('team game win', lp, ll, w2i_nb, cls)
print('predict(team game win) =', pred, '| scores:', {k: round(v,2) for k,v in sc.items()})
assert pred == 'sport', '含 team/game/win 应判为 sport'
assert predict_nb('code data model', lp, ll, w2i_nb, cls)[0] == 'tech'
print('✅ 朴素贝叶斯: 对数空间+平滑, 正确分类 sport/tech')

## 5 · 精确率 / 召回率 / F1

$P=\frac{TP}{TP+FP}$, $R=\frac{TP}{TP+FN}$, $F_1=\frac{2PR}{P+R}$。**亲眼看 accuracy 的陷阱**：不平衡数据下，全猜多数类 accuracy 很高但 F1=0。

In [ ]:
def prf1(y_true, y_pred, positive):
    '''针对 positive 类算 precision/recall/F1。'''
    TP = sum(t == positive and p == positive for t, p in zip(y_true, y_pred))
    FP = sum(t != positive and p == positive for t, p in zip(y_true, y_pred))
    FN = sum(t == positive and p != positive for t, p in zip(y_true, y_pred))
    prec = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    rec = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return prec, rec, f1

# 不平衡: 10 个样本, 只有 2 个正例(spam)
y_true = ['ham']*8 + ['spam']*2
# 傻瓜分类器: 全猜 ham(多数类)
y_dumb = ['ham']*10
acc_dumb = sum(t == p for t, p in zip(y_true, y_dumb)) / len(y_true)
p_d, r_d, f_d = prf1(y_true, y_dumb, 'spam')
print(f'傻瓜(全猜ham): accuracy={acc_dumb:.0%}  但 spam 的 P={p_d:.2f} R={r_d:.2f} F1={f_d:.2f}')
assert acc_dumb == 0.8, '全猜多数类 accuracy 高达 80%'
assert f_d == 0.0, '但一个 spam 都没找到 -> F1=0'
# 一个真正有用的分类器
y_good = ['ham']*8 + ['spam']*2
assert prf1(y_true, y_good, 'spam')[2] == 1.0
print('✅ F1 揭穿了 accuracy 的陷阱: 80% accuracy 的傻瓜分类器 F1=0')

## 6 · 完整管线 + 混淆矩阵

把分词→NB→评测串成一条管线, 在玩具训练/测试集上跑通, 算混淆矩阵与 macro-F1。

In [ ]:
def confusion_matrix(y_true, y_pred, classes):
    ci = {c: i for i, c in enumerate(classes)}
    M = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        M[ci[t], ci[p]] += 1
    return M

def macro_f1(y_true, y_pred, classes):
    return float(np.mean([prf1(y_true, y_pred, c)[2] for c in classes]))

test_docs = ['game score team', 'software code data', 'player goal game']
test_labels = ['sport', 'tech', 'sport']
preds = [predict_nb(d, lp, ll, w2i_nb, cls)[0] for d in test_docs]
print('预测:', preds)
print('真值:', test_labels)
M = confusion_matrix(test_labels, preds, cls)
print('混淆矩阵 (行=真, 列=预测), 类序', cls, ':')
print(M)
mf1 = macro_f1(test_labels, preds, cls)
print('macro-F1 =', round(mf1, 3))
assert M.trace() == sum(t == p for t, p in zip(test_labels, preds)), '对角线=正确数'
assert mf1 == 1.0, '这个玩具测试集应全对'
print('✅ 完整管线: 分词->NB->混淆矩阵->macro-F1 跑通')

---
## ✏️ 练习 1：分词与归一化

实现 `preprocess(text)`：小写化 + 只保留字母词 + 去掉长度 < 2 的词（去掉单字符噪声）。不去停用词。

In [ ]:
def preprocess(text):
    # TODO: 小写, re.findall 字母词, 过滤掉 len < 2 的
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert preprocess('Hello, A World!') == ['hello', 'world'], '去标点+去单字符 a'
assert preprocess('THE cat') == ['the', 'cat'], '小写, 不去停用词'
assert preprocess('x y zz') == ['zz'], '单字符 x,y 被过滤'
print('✅ 练习 1 通过：分词归一化')

## ✏️ 练习 2：TF-IDF

实现 `compute_idf(docs, w2i)`：返回每个词的 IDF = $\log\frac{N}{\mathrm{df}(t)}$ 数组。（df = 含该词的文档数；用 `tokenize`。）

In [ ]:
def compute_idf(docs, w2i):
    # TODO: N=文档数; df[i]=含词i的文档数; 返回 log(N/max(df,1))
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
dd = ['cat dog', 'cat bird', 'cat fish']
wi = build_vocab(dd)
idf = compute_idf(dd, wi)
# cat 在全部 3 篇 -> IDF=log(3/3)=0
assert np.isclose(idf[wi['cat']], 0.0)
# dog 只在 1 篇 -> IDF=log(3/1)>0
assert np.isclose(idf[wi['dog']], np.log(3))
assert idf[wi['dog']] > idf[wi['cat']]
print('✅ 练习 2 通过：IDF 压低普遍词')

## ✏️ 练习 3：朴素贝叶斯似然

实现 `class_log_likelihood(class_word_counts, alpha, V)`：给定一个类的词计数向量，返回拉普拉斯平滑后的对数似然 $\log\frac{c_t+\alpha}{(\sum c)+\alpha V}$。

In [ ]:
def class_log_likelihood(class_word_counts, alpha, V):
    # TODO: log((counts + alpha) / (counts.sum() + alpha*V))
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
counts = np.array([3.0, 1.0, 0.0])     # 第3个词在该类没出现
V = 3
ll = class_log_likelihood(counts, alpha=1.0, V=V)
# 手算: (3+1)/(4+3)=4/7, (1+1)/7=2/7, (0+1)/7=1/7
assert np.allclose(np.exp(ll), [4/7, 2/7, 1/7]), '平滑似然应与手算一致'
# 未出现的词(第3个)概率应 > 0 (平滑的作用)
assert np.exp(ll[2]) > 0
# 概率应归一(同一类内所有词)
assert abs(np.exp(ll).sum() - 1.0) < 1e-9
print('✅ 练习 3 通过：朴素贝叶斯平滑似然')

## ✏️ 练习 4：F1 评测

实现 `f1_score(y_true, y_pred, positive)`：返回 positive 类的 F1。$F_1=\frac{2PR}{P+R}$，分母为 0 时返回 0。

In [ ]:
def f1_score(y_true, y_pred, positive):
    # TODO: 算 TP/FP/FN -> P,R -> F1; 注意各种 0 分母
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
yt = ['pos', 'pos', 'neg', 'pos', 'neg']
yp = ['pos', 'neg', 'neg', 'pos', 'pos']
# TP=2(第1,4), FP=1(第5), FN=1(第2) -> P=2/3, R=2/3, F1=2/3
f = f1_score(yt, yp, 'pos')
assert abs(f - 2/3) < 1e-9, f'应为 2/3, 得 {f}'
# 完美预测 -> F1=1
assert f1_score(yt, yt, 'pos') == 1.0
# 全预测错(没一个pos预测对) -> F1=0
assert f1_score(['pos','pos'], ['neg','neg'], 'pos') == 0.0
print('✅ 练习 4 通过：F1 评测')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def preprocess(text):
    return [t for t in re.findall(r'[a-z]+', text.lower()) if len(t) >= 2]

In [ ]:
# 练习 2 参考答案
def compute_idf(docs, w2i):
    N = len(docs); df = np.zeros(len(w2i))
    for d in docs:
        for t in set(tokenize(d)):
            if t in w2i:
                df[w2i[t]] += 1
    return np.log(N / np.maximum(df, 1))

In [ ]:
# 练习 3 参考答案
def class_log_likelihood(class_word_counts, alpha, V):
    total = class_word_counts.sum()
    return np.log((class_word_counts + alpha) / (total + alpha * V))

In [ ]:
# 练习 4 参考答案
def f1_score(y_true, y_pred, positive):
    TP = sum(t == positive and p == positive for t, p in zip(y_true, y_pred))
    FP = sum(t != positive and p == positive for t, p in zip(y_true, y_pred))
    FN = sum(t == positive and p != positive for t, p in zip(y_true, y_pred))
    P = TP / (TP + FP) if (TP + FP) else 0.0
    R = TP / (TP + FN) if (TP + FN) else 0.0
    return 2 * P * R / (P + R) if (P + R) else 0.0

---
## 🧪 真实数据胶囊：真实文本情感/主题分类

用真实风格的短文本分类语料（**内置真实英文影评/科技短句**），训练朴素贝叶斯，在测试集上算 accuracy 与 macro-F1，对比二者。

In [ ]:
def load_text_classification():
    '''真实风格短文本 + 标签。返回 (docs, labels)。'''
    # 真实英文短句: pos=正面影评, neg=负面影评
    data = [
        ('this movie was great and the acting was wonderful', 'pos'),
        ('an excellent film with a brilliant story', 'pos'),
        ('i loved this movie it was fantastic and fun', 'pos'),
        ('a wonderful and beautiful film truly great', 'pos'),
        ('terrible movie the plot was boring and bad', 'neg'),
        ('awful film i hated the acting it was horrible', 'neg'),
        ('a boring and dull movie waste of time', 'neg'),
        ('the worst film bad acting and terrible story', 'neg'),
    ]
    return data, 'builtin real movie-review sentiment'

data, src = load_text_classification()
print('数据来源:', src, '| 样本:', len(data))
cap_docs = [d for d, _ in data]
cap_labels = [l for _, l in data]
# 留出测试: 每类各取最后一个
cap_train = [(d, l) for d, l in data[:3]] + [(d, l) for d, l in data[4:7]]
cap_test = [data[3], data[7]]
print('训练:', len(cap_train), '| 测试:', len(cap_test))
assert len(cap_train) > 0 and len(cap_test) > 0
print('✅ 真实风格情感分类语料就绪')

**🧪 胶囊练习**：用 `train_naive_bayes` 训练, 在测试集预测, 算 macro-F1。补全调用。

In [ ]:
tr_docs = [d for d, _ in cap_train]; tr_labels = [l for _, l in cap_train]
te_docs = [d for d, _ in cap_test]; te_labels = [l for _, l in cap_test]
# TODO: lp2, ll2, w2i2c, cls2 = train_naive_bayes(tr_docs, tr_labels)
#       preds2 = [predict_nb(d, lp2, ll2, w2i2c, cls2)[0] for d in te_docs]
#       mf1_cap = macro_f1(te_labels, preds2, cls2)
raise NotImplementedError

In [ ]:
# 自测
assert 0.0 <= mf1_cap <= 1.0
acc_cap = sum(t == p for t, p in zip(te_labels, preds2)) / len(te_labels)
print(f'情感分类: accuracy={acc_cap:.0%}  macro-F1={mf1_cap:.2f}')
assert mf1_cap >= 0.5, '朴素贝叶斯在这种语料上应不差'
print('✅ 胶囊练习通过：真实文本情感分类管线')

In [ ]:
# 📖 胶囊参考答案
tr_docs = [d for d, _ in cap_train]; tr_labels = [l for _, l in cap_train]
te_docs = [d for d, _ in cap_test]; te_labels = [l for _, l in cap_test]
lp2, ll2, w2i2c, cls2 = train_naive_bayes(tr_docs, tr_labels)
preds2 = [predict_nb(d, lp2, ll2, w2i2c, cls2)[0] for d in te_docs]
mf1_cap = macro_f1(te_labels, preds2, cls2)
print('预测:', preds2, '| 真值:', te_labels, '| macro-F1:', round(mf1_cap, 3))

### 小结
- 文本分类管线 4 步: **分词归一化 → TF-IDF → 分类器(NB/logreg) → P/R/F1**。
- **TF-IDF** = 词频 × 逆文档频率, 自动压低 the/of(IDF≈0)、抬高有区分度的词。
- **朴素贝叶斯** 生成式、数频率+平滑、对数空间, 极快的强基线; **逻辑回归** 判别式、学权重、常更准(=单标签 CRF)。
- **accuracy 在不平衡数据下骗人**(全猜多数类也高); 必看 **P/R/F1**(调和平均, 惩罚极端)。
- **铁律**: 训练测试用同一预处理与向量化器(只在训练集 fit), 否则数据泄漏。
- **总纲**: 五模块覆盖 表示(01)→语言模型(02)→序列标注(03/04)→分类(05), 全是现代 NLP 的地基。

🎉 **恭喜读完整门课!** 你已能从零写出 word2vec / n-gram / HMM / CRF / TF-IDF —— 现代 NLP 的本质对你不再是黑盒。